In [1]:
import sys
from pathlib import Path
from pyprojroot import here

sys.path.append(str(here()))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.processing import BaselineTransformer, build_pipeline
from sklearn.model_selection import RepeatedStratifiedKFold
from lightgbm import LGBMRegressor
from src.train import cv_result
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import skew
from src.config import cfg
from src.utils import set_seed, load_data
from typing import ClassVar
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

from src.processing import MemoryOptimizer, ToCategory
from src.train import cv_result

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
gseed = cfg.general.seed
set_seed(gseed)


In [5]:
train_path = Path(cfg.paths.train)
test_path = Path(cfg.paths.test)

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

In [6]:
X_train, y_train_raw, X_test, test_ids = load_data(cfg, False)

y_train = np.log1p(y_train_raw)

y_binned = pd.qcut(y_train, q=10, labels=False)

rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=gseed)

In [7]:
class FinalTransformer(BaseEstimator, TransformerMixin):
    NONE_COLS: ClassVar[list[str]] = [
        "PoolQC",
        "MiscFeature",
        "Alley",
        "Fence",
        "FireplaceQu",
        "MasVnrType",
        "GarageQual",
        "GarageFinish",
        "GarageType",
        "GarageCond",
        "BsmtQual",
        "BsmtCond",
        "BsmtExposure",
        "BsmtFinType1",
        "BsmtFinType2",
    ]

    ZERO_COLS: ClassVar[list[str]] = [
        "MasVnrArea",
        "GarageYrBlt",
        "GarageArea",
        "GarageCars",
        "BsmtFinSF1",
        "BsmtFinSF2",
        "BsmtUnfSF",
        "TotalBsmtSF",
        "BsmtFullBath",
        "BsmtHalfBath",
    ]

    MODE_COLS: ClassVar[list[str]] = [
        "Electrical",
        "MSZoning",
        "Exterior1st",
        "Exterior2nd",
        "SaleType",
        "KitchenQual",
    ]

    CONTINUOUS_CANDIDATES: ClassVar[list[str]] = [
        "LotFrontage",
        "LotArea",
        "OverallQual",
        "OverallCond",
        "MasVnrArea",
        "BsmtFinSF1",
        "BsmtFinSF2",
        "BsmtUnfSF",
        "TotalBsmtSF",
        "1stFlrSF",
        "2ndFlrSF",
        "GrLivArea",
        "BsmtFullBath",
        "FullBath",
        "HalfBath",
        "BedroomAbvGr",
        "KitchenAbvGr",
        "TotRmsAbvGrd",
        "Fireplaces",
        "GarageCars",
        "GarageArea",
        "WoodDeckSF",
        "OpenPorchSF",
        "EnclosedPorch",
        "ScreenPorch",
        "PoolArea",
        "MoSold",
        "YrSold",
        "HouseAge",
        "RemodAge",
        "GarageAge",
        "TotalSF",
        "TotalBathrooms",
        "TotalPorchSF",
        "AreaPerRoom",
        "LivingAreaRatio",
    ]

    def _fill_none_and_zero(self, X):
        X = X.copy()

        for col in self.NONE_COLS:
            X[col] = X[col].fillna("None")

        for col in self.ZERO_COLS:
            X[col] = X[col].fillna(0)

        return X

    def _engineer_features(self, X):
        X = X.copy()

        X["MoSold_sin"] = np.sin(2 * np.pi * X["MoSold"] / 12)
        X["MoSold_cos"] = np.cos(2 * np.pi * X["MoSold"] / 12)

        X["HouseAge"] = X["YrSold"] - X["YearBuilt"]
        X["RemodAge"] = X["YrSold"] - X["YearRemodAdd"]
        X["GarageAge"] = X["YrSold"] - X["GarageYrBlt"]

        X.loc[X["GarageType"] == "None", "GarageAge"] = 0

        X["IsRemodeled"] = (X["YearBuilt"] != X["YearRemodAdd"]).astype(int)

        X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
        X["TotalPorchSF"] = (
            X["OpenPorchSF"] + X["EnclosedPorch"] + X["3SsnPorch"] + X["ScreenPorch"]
        )
        X["TotalBathrooms"] = (
            X["FullBath"]
            + 0.5 * X["HalfBath"]
            + X["BsmtFullBath"]
            + 0.5 * X["BsmtHalfBath"]
        )

        X["AreaPerRoom"] = X["GrLivArea"] / X["TotRmsAbvGrd"].replace(0, np.nan)
        X["LivingAreaRatio"] = X["GrLivArea"] / X["LotArea"].replace(0, np.nan)

        return X

    def fit(self, X_original, y=None):
        X = X_original.copy()

        self.lotfrontage_medians2_ = X.groupby(by=["Neighborhood", "LotConfig"])[
            "LotFrontage"
        ].median()

        self.lotfrontage_medians1_ = X.groupby(by=["Neighborhood"])[
            "LotFrontage"
        ].median()

        self.lotfrontage_global_median_ = X["LotFrontage"].median()

        self.modes_ = {col: X[col].mode()[0] for col in self.MODE_COLS}

        X = self._fill_none_and_zero(X)
        X = self._engineer_features(X)

        skewed = X[self.CONTINUOUS_CANDIDATES].apply(lambda x: skew(x.dropna()))
        self.skewed_cols_ = skewed[abs(skewed) > 0.85].index.tolist()

        return self

    def transform(self, X_original):
        X = X_original.copy()

        X = self._fill_none_and_zero(X)

        keys2 = list(zip(X["Neighborhood"], X["LotConfig"]))
        filled_2 = pd.Series(keys2, index=X.index).map(self.lotfrontage_medians2_)
        X["LotFrontage"] = X["LotFrontage"].fillna(filled_2)

        filled_1 = X["Neighborhood"].map(self.lotfrontage_medians1_)
        X["LotFrontage"] = X["LotFrontage"].fillna(filled_1)

        X["LotFrontage"] = X["LotFrontage"].fillna(self.lotfrontage_global_median_)

        for col, mode_value in self.modes_.items():
            X[col] = X[col].fillna(mode_value)

        X["Functional"] = X["Functional"].fillna("Typ")

        if "MSSubClass" in X.columns:
            X["MSSubClass"] = X["MSSubClass"].astype(str)

        X = self._engineer_features(X)

        for col in [
            "Id",
            "Utilities",
            "3SsnPorch",
            "BsmtHalfBath",
            "Condition2",
            "GarageYrBlt",
            "Heating",
            "LowQualFinSF",
            "MiscVal",
            "PoolQC",
            "RoofMatl",
            "Street",
            "YearBuilt",
            "YearRemodAdd",
        ]:
            if col in X.columns:
                X = X.drop(columns=col)

        for col in self.skewed_cols_:
            if col in X.columns:
                X[col] = np.log1p(X[col])

        return X

In [8]:
num_columns = [
    "LotFrontage",
    "LotArea",
    "OverallQual",
    "OverallCond",
    # "YearBuilt",        # удалено -> заменено на HouseAge
    # "YearRemodAdd",     # удалено -> заменено на RemodAge
    "MasVnrArea",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    # "LowQualFinSF",   # исключено - useless_features (permutation importance)
    "GrLivArea",
    "BsmtFullBath",
    # "BsmtHalfBath",   # исключено - useless_features (permutation importance)
    "FullBath",
    "HalfBath",
    "BedroomAbvGr",
    "KitchenAbvGr",
    "TotRmsAbvGrd",
    "Fireplaces",
    # "GarageYrBlt",      # удалено -> заменено на GarageAge
    "GarageCars",
    "GarageArea",
    "WoodDeckSF",
    "OpenPorchSF",
    "EnclosedPorch",
    # "3SsnPorch",      # исключено - useless_features (permutation importance)
    "ScreenPorch",
    "PoolArea",  # лучше пусть останется упоминание о бассейне
    # "MiscVal",        # исключено - useless_features (permutation importance)
    "MoSold",
    "YrSold",
    # --- новые непрерывные/счётные признаки ---
    "HouseAge",
    "RemodAge",
    "GarageAge",
    "TotalSF",
    "TotalBathrooms",
    "TotalPorchSF",
    "AreaPerRoom",
    "LivingAreaRatio",
    "MoSold_sin",
    "MoSold_cos",
    # --- бинарные флаги (0/1) ---
    "IsRemodeled",
    # "IsNewHouse",   # исключено - useless_features
    # "HasPool",      # исключено - useless_features
    # "HasGarage",    # исключено - useless_features
    # "HasBsmt",      # исключено - useless_features
    # "HasFireplace", # исключено - useless_features
    # "Has2ndFloor",  # исключено - useless_features
]

ordinal_columns = [
    "LotShape",
    "LandSlope",
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "HeatingQC",
    "KitchenQual",
    "Functional",
    "FireplaceQu",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PavedDrive",
    # "PoolQC",   # исключено - useless_features
    "CentralAir",
]

ordinal_categories = [
    # порядок соответствует ordinal_columns выше (PoolQC и его категория удалены)
    ["IR3", "IR2", "IR1", "Reg"],
    ["Sev", "Mod", "Gtl"],
    ["Po", "Fa", "TA", "Gd", "Ex"],
    ["Po", "Fa", "TA", "Gd", "Ex"],
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    ["None", "No", "Mn", "Av", "Gd"],
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    ["Po", "Fa", "TA", "Gd", "Ex"],
    ["Po", "Fa", "TA", "Gd", "Ex"],
    ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    ["None", "Unf", "RFn", "Fin"],
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    ["N", "P", "Y"],
    # ["None", "Fa", "TA", "Gd", "Ex"],  # было для PoolQC - удалено
    ["N", "Y"],
]

ohe_columns = [
    "Fence",
    "Electrical",
    "MSSubClass",
    "MSZoning",
    # "Street",        # исключено - useless_features
    "Alley",
    "LandContour",
    "LotConfig",
    "Neighborhood",
    "Condition1",
    # "Condition2",    # исключено - useless_features
    "BldgType",
    "HouseStyle",
    "RoofStyle",
    # "RoofMatl",      # исключено - useless_features
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    # "Heating",       # исключено - useless_features
    "GarageType",
    "MiscFeature",
    "SaleType",
    "SaleCondition",
]
tree_pipe = Pipeline(
    [
        ("processor", FinalTransformer()),
        ("memory_optimizer", MemoryOptimizer()),
        (
            "column_transformer",
            ColumnTransformer(
                transformers=[
                    ("num_features", "passthrough", num_columns),
                    (
                        "cat_features_oe",
                        OrdinalEncoder(
                            categories=ordinal_categories,
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                        ordinal_columns,
                    ),
                    (
                        "cat_features_ohe",
                        "passthrough",
                        ohe_columns,
                    ),
                ],
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
        ("to_category", ToCategory(ohe_columns)),
    ]
)

from sklearn.preprocessing import StandardScaler

lin_pipe = Pipeline(
    [
        ("processor", FinalTransformer()),
        ("memory_optimizer", MemoryOptimizer()),
        (
            "column_transformer",
            ColumnTransformer(
                transformers=[
                    (
                        "num_features",
                        StandardScaler(),
                        num_columns,
                    ),
                    (
                        "cat_features_oe",
                        Pipeline(
                            [
                                (
                                    "encode",
                                    OrdinalEncoder(
                                        categories=ordinal_categories,
                                        handle_unknown="use_encoded_value",
                                        unknown_value=-1,
                                    ),
                                ),
                                ("scale", StandardScaler()),
                            ]
                        ),
                        ordinal_columns,
                    ),
                    (
                        "cat_features_ohe",
                        OneHotEncoder(
                            drop="first",
                            handle_unknown="infrequent_if_exist",
                            min_frequency=10,
                            sparse_output=False,
                        ),
                        ohe_columns,
                    ),
                ],
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
    ]
)

assert len(ordinal_columns) == len(ordinal_categories), (
    "Рассинхрон между колонками и категориями!"
)

### baseline

In [9]:
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from catboost import CatBoostRegressor

In [19]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    LGBMRegressor(random_state=gseed, verbosity=-1),
    X_train,
    y_train,
    y_binned,
    rskf,
    tree_pipe,
    "baseline_model",
)
metrics

baseline_model:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,baseline_model,0.03689,0.000739,0.123829,0.012789,0.120986


In [35]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    LinearRegression(), X_train, y_train, y_binned, rskf, lin_pipe, "lin_reg"
)
metrics

lin_reg:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,lin_reg,0.093288,0.00143,0.11321,0.013364,0.112858


In [73]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    Ridge(), X_train, y_train, y_binned, rskf, lin_pipe, "ridge_reg"
)
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,ridge_reg,0.0938,0.001435,0.110749,0.013143,0.110652


In [ ]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    Lasso(alpha=0.01), X_train, y_train, y_binned, rskf, lin_pipe, "lasso_reg"
)
metrics

lasso_reg:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,lasso_reg,0.119912,0.001395,0.122319,0.012302,0.122706


In [ ]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    ElasticNet(alpha=0.01), X_train, y_train, y_binned, rskf, lin_pipe, "elastic_net"
)
metrics

elastic_net:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,elastic_net,0.116877,0.001417,0.120102,0.012315,0.120561


In [88]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    KNeighborsRegressor(), X_train, y_train, y_binned, rskf, lin_pipe, "knn_regressor"
)
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,knn_regressor,0.128849,0.001532,0.158134,0.01347,0.156096


In [89]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    SVR(C=0.5), X_train, y_train, y_binned, rskf, lin_pipe, "svr"
)
metrics

svr:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,svr,0.088035,0.001245,0.140214,0.01796,0.140826


In [ ]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    RandomForestRegressor(random_state=gseed),
    X_train,
    y_train,
    y_binned,
    rskf,
    lin_pipe,
    "random_forest",
)
metrics

random_forest:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,random_forest,0.051303,0.000824,0.136488,0.012877,0.135401


In [ ]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    GradientBoostingRegressor(),
    X_train,
    y_train,
    y_binned,
    rskf,
    lin_pipe,
    "sklearn_gbdt",
)
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,sklearn_gbdt,0.077669,0.001202,0.120881,0.013783,0.119422


In [ ]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    HistGradientBoostingRegressor(),
    X_train,
    y_train,
    y_binned,
    rskf,
    lin_pipe,
    "sklearn_hist_gbdt",
)
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,sklearn_hist_gbdt,0.039042,0.000877,0.125394,0.012208,0.122598


In [ ]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    HistGradientBoostingRegressor(),
    X_train,
    y_train,
    y_binned,
    rskf,
    lin_pipe,
    "sklearn_hist_gbdt",
)
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,sklearn_hist_gbdt,0.039042,0.000877,0.125394,0.012208,0.122598


In [25]:
fitted_models, oof_preds, metrics, val_scores = cv_result(
    CatBoostRegressor(random_state=gseed, loss_function="RMSE"),
    X_train,
    y_train,
    y_binned,
    rskf,
    tree_pipe,
    fit_params={"model__cat_features": list(ohe_columns)},
)
metrics

CV folds:   0%|          | 0/50 [00:00<?, ?fold/s]

Learning rate set to 0.042738
0:	learn: 0.3875577	total: 22.6ms	remaining: 22.6s
1:	learn: 0.3769581	total: 48.9ms	remaining: 24.4s
2:	learn: 0.3666456	total: 82.4ms	remaining: 27.4s
3:	learn: 0.3570172	total: 122ms	remaining: 30.3s
4:	learn: 0.3474048	total: 149ms	remaining: 29.6s
5:	learn: 0.3383886	total: 173ms	remaining: 28.7s
6:	learn: 0.3297794	total: 197ms	remaining: 27.9s
7:	learn: 0.3211660	total: 224ms	remaining: 27.8s
8:	learn: 0.3131202	total: 250ms	remaining: 27.6s
9:	learn: 0.3049609	total: 274ms	remaining: 27.1s
10:	learn: 0.2970141	total: 299ms	remaining: 26.9s
11:	learn: 0.2893075	total: 325ms	remaining: 26.7s
12:	learn: 0.2829105	total: 352ms	remaining: 26.7s
13:	learn: 0.2767194	total: 380ms	remaining: 26.7s
14:	learn: 0.2700917	total: 404ms	remaining: 26.6s
15:	learn: 0.2642530	total: 430ms	remaining: 26.4s
16:	learn: 0.2583854	total: 455ms	remaining: 26.3s
17:	learn: 0.2524203	total: 483ms	remaining: 26.3s
18:	learn: 0.2471084	total: 510ms	remaining: 26.4s
19:	lear

d:\vs_projects\fp_houses\src\processing.py:179: UserWarning: Column 'Exterior1st': 1 unseen categories converted to Unseen on transform.
  
CV folds:   2%|▏         | 1/50 [00:30<25:01, 30.64s/fold]

Learning rate set to 0.042738
0:	learn: 0.3869299	total: 25.5ms	remaining: 25.4s
1:	learn: 0.3764993	total: 55.2ms	remaining: 27.5s
2:	learn: 0.3658834	total: 84.1ms	remaining: 27.9s
3:	learn: 0.3554544	total: 113ms	remaining: 28.1s
4:	learn: 0.3468280	total: 140ms	remaining: 27.8s
5:	learn: 0.3371119	total: 167ms	remaining: 27.7s
6:	learn: 0.3281173	total: 200ms	remaining: 28.3s
7:	learn: 0.3194937	total: 230ms	remaining: 28.6s
8:	learn: 0.3110526	total: 264ms	remaining: 29s
9:	learn: 0.3030352	total: 295ms	remaining: 29.2s
10:	learn: 0.2960080	total: 324ms	remaining: 29.1s
11:	learn: 0.2889917	total: 354ms	remaining: 29.2s
12:	learn: 0.2825443	total: 384ms	remaining: 29.2s
13:	learn: 0.2756549	total: 412ms	remaining: 29s
14:	learn: 0.2689565	total: 441ms	remaining: 29s
15:	learn: 0.2632580	total: 472ms	remaining: 29s
16:	learn: 0.2572140	total: 503ms	remaining: 29.1s
17:	learn: 0.2517081	total: 532ms	remaining: 29s
18:	learn: 0.2459517	total: 564ms	remaining: 29.1s
19:	learn: 0.24093

d:\vs_projects\fp_houses\src\processing.py:179: UserWarning: Column 'Neighborhood': 2 unseen categories converted to Unseen on transform.
  
d:\vs_projects\fp_houses\src\processing.py:179: UserWarning: Column 'Exterior1st': 1 unseen categories converted to Unseen on transform.
  
CV folds:   4%|▍         | 2/50 [01:02<25:00, 31.26s/fold]

...

998:	learn: 0.0436862	total: 30.8s	remaining: 30.8ms
999:	learn: 0.0436790	total: 30.8s	remaining: 0us
Learning rate set to 0.042743
0:	learn: 0.3894674	total: 25.9ms	remaining: 25.8s
1:	learn: 0.3789671	total: 57ms	remaining: 28.4s
2:	learn: 0.3679164	total: 86.3ms	remaining: 28.7s
3:	learn: 0.3579922	total: 112ms	remaining: 27.8s
4:	learn: 0.3482664	total: 140ms	remaining: 27.8s
5:	learn: 0.3387177	total: 166ms	remaining: 27.6s
6:	learn: 0.3294350	total: 193ms	remaining: 27.4s
7:	learn: 0.3216742	total: 220ms	remaining: 27.3s
8:	learn: 0.3132571	total: 248ms	remaining: 27.3s
9:	learn: 0.3058107	total: 276ms	remaining: 27.3s
10:	learn: 0.2978242	total: 305ms	remaining: 27.4s
11:	learn: 0.2906403	total: 330ms	remaining: 27.2s
12:	learn: 0.2839296	total: 358ms	remaining: 27.2s
13:	learn: 0.2771435	total: 385ms	remaining: 27.1s
14:	learn: 0.2713289	total: 412ms	remaining: 27.1s
15:	learn: 0.2649618	total: 441ms	remaining: 27.1s
16:	learn: 0.2593160	total: 473ms	remaining: 27.4s
17:	learn

d:\vs_projects\fp_houses\src\processing.py:179: UserWarning: Column 'Exterior2nd': 1 unseen categories converted to Unseen on transform.
  
d:\vs_projects\fp_houses\src\processing.py:179: UserWarning: Column 'Electrical': 1 unseen categories converted to Unseen on transform.
  
CV folds:  98%|█████████▊| 49/50 [25:46<00:31, 31.21s/fold]

Learning rate set to 0.042743
0:	learn: 0.3885175	total: 24.7ms	remaining: 24.7s
1:	learn: 0.3780591	total: 54.6ms	remaining: 27.3s
2:	learn: 0.3675865	total: 84.9ms	remaining: 28.2s
3:	learn: 0.3581650	total: 113ms	remaining: 28.2s
4:	learn: 0.3488432	total: 140ms	remaining: 27.9s
5:	learn: 0.3398063	total: 166ms	remaining: 27.5s
6:	learn: 0.3312130	total: 192ms	remaining: 27.3s
7:	learn: 0.3229641	total: 224ms	remaining: 27.8s
8:	learn: 0.3144882	total: 266ms	remaining: 29.3s
9:	learn: 0.3065310	total: 298ms	remaining: 29.5s
10:	learn: 0.2992427	total: 324ms	remaining: 29.1s
11:	learn: 0.2923287	total: 351ms	remaining: 28.9s
12:	learn: 0.2852628	total: 377ms	remaining: 28.6s
13:	learn: 0.2786314	total: 405ms	remaining: 28.5s
14:	learn: 0.2720618	total: 432ms	remaining: 28.4s
15:	learn: 0.2658464	total: 463ms	remaining: 28.5s
16:	learn: 0.2602328	total: 491ms	remaining: 28.4s
17:	learn: 0.2545301	total: 522ms	remaining: 28.5s
18:	learn: 0.2491095	total: 550ms	remaining: 28.4s
19:	lear

d:\vs_projects\fp_houses\src\processing.py:179: UserWarning: Column 'MiscFeature': 2 unseen categories converted to Unseen on transform.
  


,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,None,0.051251,0.001906,0.113443,0.013345,0.11239


### tuning

* elastic net
* lgbm
* catboost

In [10]:
from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error
import optuna

In [11]:
def run_one_fold(
    pipe,
    build_model_fn,
    params,
    X_tr,
    y_tr,
    X_val,
    y_val,
    fit_extra_fn=None,
    early_stopping=True,
):

    prep = clone(pipe)
    X_tr_t = prep.fit_transform(X_tr)
    X_val_t = prep.transform(X_val)

    model = build_model_fn(params)
    fit_kwargs = dict(fit_extra_fn(params) if fit_extra_fn else {})
    if early_stopping:
        fit_kwargs["eval_set"] = [(X_val_t, y_val)]

    model.fit(X_tr_t, y_tr, **fit_kwargs)

    rmse = root_mean_squared_error(y_val, model.predict(X_val_t))
    fitted_pipe = Pipeline([("preprocessor", prep), ("model", model)])
    return rmse, fitted_pipe

In [12]:
from tqdm.auto import tqdm

In [13]:
def make_objective(
    sample_params_fn,
    build_model_fn,
    pipe,
    X,
    y,
    y_binned,
    cv,
    fit_extra_fn=None,
    early_stopping=True,
):
    def objective(trial):
        params = sample_params_fn(trial)
        fold_rmses = []
        fold_iter = enumerate(cv.split(X, y_binned))
        for step, (tr_idx, val_idx) in tqdm(
            fold_iter,
            total=cv.get_n_splits(),
            desc=f"trial {trial.number}",
            leave=False,
        ):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

            rmse, _ = run_one_fold(
                pipe,
                build_model_fn,
                params,
                X_tr,
                y_tr,
                X_val,
                y_val,
                fit_extra_fn=fit_extra_fn,
                early_stopping=early_stopping,
            )
            fold_rmses.append(rmse)

            trial.report(float(np.mean(fold_rmses)), step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()

        trial.set_user_attr("fold_std", float(np.std(fold_rmses)))
        return float(np.mean(fold_rmses))

    return objective

In [14]:
def run_confirm_cv(
    pipe,
    build_model_fn,
    params,
    X,
    y,
    y_binned,
    cv,
    fit_extra_fn=None,
    early_stopping=True,
):
    fold_rmses, fitted_pipes = [], []
    for tr_idx, val_idx in tqdm(
        cv.split(X, y_binned), total=cv.get_n_splits(), desc="confirm cv"
    ):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        rmse, fitted_pipe = run_one_fold(
            pipe,
            build_model_fn,
            params,
            X_tr,
            y_tr,
            X_val,
            y_val,
            fit_extra_fn=fit_extra_fn,
            early_stopping=early_stopping,
        )
        fold_rmses.append(rmse)
        fitted_pipes.append(fitted_pipe)
    return fold_rmses, fitted_pipes

In [15]:
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold

X_opt, X_hold, y_opt, y_hold, yb_opt, yb_hold = train_test_split(
    X_train,
    y_train,
    y_binned,
    test_size=0.15,
    stratify=y_binned,
    random_state=gseed,
)

yb_opt = pd.qcut(y_opt, q=10, labels=False)

rskf_search = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=gseed)
rskf_confirm = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=gseed)

In [16]:
import lightgbm as lgb


def sample_params_en(trial):
    return {
        "alpha": trial.suggest_float("alpha", 1e-4, 10.0, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
    }


def build_en(params):
    return ElasticNet(random_state=gseed, max_iter=20000, **params)


def sample_params_lgbm(trial):
    return {
        "num_leaves": trial.suggest_int("num_leaves", 7, 255, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }


def build_lgbm(params):
    return LGBMRegressor(random_state=gseed, verbosity=-1, n_estimators=5000, **params)


def lgbm_fit_extra(params=None):
    return {"callbacks": [lgb.early_stopping(100, verbose=False)]}


def sample_params_cb(trial):
    return {
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
    }


def build_cb(params):
    return CatBoostRegressor(
        random_state=gseed,
        loss_function="RMSE",
        iterations=5000,
        thread_count=1,
        **params,
    )


def cb_fit_extra(params=None):
    return {
        "cat_features": list(ohe_columns),
        "early_stopping_rounds": 100,
        "use_best_model": True,
        "verbose": False,
    }


In [ ]:
import os
import optuna

output_dir = "optuna_results"
os.makedirs(output_dir, exist_ok=True)

storage = f"sqlite:///{output_dir}/optuna_studies.db"


sampler_en = optuna.samplers.TPESampler(seed=gseed, multivariate=True)
sampler_lgbm = optuna.samplers.TPESampler(seed=gseed, multivariate=True)
sampler_cb = optuna.samplers.TPESampler(seed=gseed, multivariate=True)

pruner_lgbm = optuna.pruners.MedianPruner(
    n_startup_trials=10, n_warmup_steps=3, interval_steps=1
)
pruner_cb = optuna.pruners.MedianPruner(
    n_startup_trials=10, n_warmup_steps=3, interval_steps=1
)

study_en = optuna.create_study(
    study_name="elastic_net",
    direction="minimize",
    sampler=sampler_en,
    storage=storage,
    load_if_exists=True,
)
study_lgbm = optuna.create_study(
    study_name="lightgbm",
    direction="minimize",
    sampler=sampler_lgbm,
    pruner=pruner_lgbm,
    storage=storage,
    load_if_exists=True,
)
study_cb = optuna.create_study(
    study_name="catboost",
    direction="minimize",
    sampler=sampler_cb,
    pruner=pruner_cb,
    storage=storage,
    load_if_exists=True,
)


[I 2026-09-14 01:01:35,694] Using an existing study with `study_name='elastic_net'` instead of creating a new one.
[I 2026-09-14 01:01:35,727] Using an existing study with `study_name='lightgbm'` instead of creating a new one.
[I 2026-09-14 01:01:35,756] Using an existing study with `study_name='catboost'` instead of creating a new one.


In [21]:
study_en.optimize(
    make_objective(
        sample_params_en,
        build_en,
        lin_pipe,
        X_opt,
        y_opt,
        yb_opt,
        rskf_confirm,
        early_stopping=False,
    ),
    n_trials=200,
    n_jobs=4,
)

study_lgbm.optimize(
    make_objective(
        sample_params_lgbm,
        build_lgbm,
        tree_pipe,
        X_opt,
        y_opt,
        yb_opt,
        rskf_search,
        fit_extra_fn=lgbm_fit_extra,
    ),
    n_trials=80,
    n_jobs=1,
)

study_cb.optimize(
    make_objective(
        sample_params_cb,
        build_cb,
        tree_pipe,
        X_opt,
        y_opt,
        yb_opt,
        rskf_search,
        fit_extra_fn=cb_fit_extra,
    ),
    n_trials=60,
    n_jobs=1,
)

d:\vs_projects\fp_houses\.venv\Lib\site-packages\sqlalchemy\orm\state.py:204: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001604B1B4400>
  self.obj = weakref.ref(obj, self._cleanup)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sqlalchemy\orm\state.py:204: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001604B1B62F0>
  self.obj = weakref.ref(obj, self._cleanup)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sqlalchemy\orm\state.py:204: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001604B0B45E0>
  self.obj = weakref.ref(obj, self._cleanup)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sqlalchemy\orm\state.py:204: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001604B1048B0>
  self.obj = weakref.ref(obj, self._cleanup)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sqlalchemy\orm\state.py:204: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x00000160

trial 476:   0%|          | 0/50 [00:00<?, ?it/s]

trial 478:   0%|          | 0/50 [00:00<?, ?it/s]

trial 479:   0%|          | 0/50 [00:00<?, ?it/s]

trial 477:   0%|          | 0/50 [00:00<?, ?it/s]

...

trial 42:   0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-09-14 08:28:21,797] Trial 42 finished with value: 0.11260360067115201 and parameters: {'depth': 4, 'learning_rate': 0.01966356272888405, 'l2_leaf_reg': 2.362625214598672, 'random_strength': 2.0165371906675125, 'bagging_temperature': 0.09359726747002999, 'border_count': 213}. Best is trial 30 with value: 0.111155043463527.


trial 43:   0%|          | 0/10 [00:00<?, ?it/s]

[W 2026-09-14 08:32:10,273] Trial 43 failed with parameters: {'depth': 4, 'learning_rate': 0.012150474034779785, 'l2_leaf_reg': 5.407177857765772, 'random_strength': 0.8468801316196507, 'bagging_temperature': 0.0718088737238704, 'border_count': 223} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "d:\vs_projects\fp_houses\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\kochn\AppData\Local\Temp\ipykernel_24964\1285715589.py", line 25, in objective
    rmse, _ = run_one_fold(
              ~~~~~~~~~~~~^
        pipe,
        ^^^^^
    ...<7 lines>...
        early_stopping=early_stopping,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\kochn\AppData\Local\Temp\ipykernel_24964\2880402208.py", line 22, in run_one_fold
    model.fit(X_tr_t, y_tr, **fit_kwargs)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vs_projects\fp_houses\.venv\Lib\s

KeyboardInterrupt: 

In [22]:
best_params = study_lgbm.best_params
rmses, fitted_pipes = run_confirm_cv(
    tree_pipe,
    build_lgbm,
    best_params,
    X_opt,
    y_opt,
    yb_opt,
    rskf_confirm,
    fit_extra_fn=lgbm_fit_extra,
)

hold_pred = np.mean([p.predict(X_hold) for p in fitted_pipes], axis=0)
hold_rmse = root_mean_squared_error(y_hold, hold_pred)
print(f"CV: {np.mean(rmses):.4f} ± {np.std(rmses):.4f}   Holdout: {hold_rmse:.4f}")

confirm cv:   0%|          | 0/50 [00:00<?, ?it/s]

d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, ev

CV: 0.1122 ± 0.0154   Holdout: 0.1257


In [ ]:
# слишком долго, лучше провернуть трюк с одной моделью
best_params = study_cb.best_params
rmses, fitted_pipes = run_confirm_cv(
    tree_pipe,
    build_cb,
    best_params,
    X_opt,
    y_opt,
    yb_opt,
    rskf_confirm,
    fit_extra_fn=cb_fit_extra,
)

hold_pred = np.mean([p.predict(X_hold) for p in fitted_pipes], axis=0)
hold_rmse = root_mean_squared_error(y_hold, hold_pred)
print(f"CV: {np.mean(rmses):.4f} ± {np.std(rmses):.4f}   Holdout: {hold_rmse:.4f}")

confirm cv:   0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [24]:
best_params = study_en.best_params
rmses, fitted_pipes = run_confirm_cv(
    lin_pipe,
    build_en,
    best_params,
    X_opt,
    y_opt,
    yb_opt,
    rskf_confirm,
    early_stopping=False,
)

hold_pred = np.mean([p.predict(X_hold) for p in fitted_pipes], axis=0)
hold_rmse = root_mean_squared_error(y_hold, hold_pred)
print(f"CV: {np.mean(rmses):.4f} ± {np.std(rmses):.4f}   Holdout: {hold_rmse:.4f}")

confirm cv:   0%|          | 0/50 [00:00<?, ?it/s]

CV: 0.1082 ± 0.0123   Holdout: 0.1127
